# Notebook 01 — Chuẩn bị dữ liệu ViMD cho Kaggle Input

Notebook stream revision ViMD đã pin từ Hugging Face đúng một lần, chuẩn hóa audio thành
WAV mono 16 kHz PCM-16, tạo tokenizer và PhoBERT cache. WAV sau đó được đóng vào một số
ZIP shard **không nén** để Notebook 02 đọc ngẫu nhiên trực tiếp từ `/kaggle/input`, không
giải nén sang `/kaggle/working`.

> Bật Internet và một GPU. Chỉ Save Version khi cell cuối báo `PASS`.


In [ ]:
# 1. Kiểm tra môi trường
!nvidia-smi
import shutil
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("Disk /kaggle/working:", shutil.disk_usage("/kaggle/working"))
assert torch.cuda.is_available(), "Cần GPU cho bước tạo PhoBERT teacher cache"


In [ ]:
# 2. Nạp source và cài dependencies
%cd /kaggle/working
%env HF_HOME=/kaggle/working/.cache/huggingface
%env HF_DATASETS_CACHE=/kaggle/working/.cache/huggingface/datasets
%env TRANSFORMERS_CACHE=/kaggle/working/.cache/huggingface/hub
%env TORCH_HOME=/kaggle/working/.cache/torch

import hashlib
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

work_dir = Path("/kaggle/working/vi_asr_project")
preferred_zip = Path("/kaggle/input/vi-asr-code/vi_asr_project.zip")
preferred_project = Path("/kaggle/input/vi-asr-code/vi_asr_project")
zip_candidates = sorted(Path("/kaggle/input").rglob("vi_asr_project.zip"))
project_candidates = sorted(
    path.parent for path in Path("/kaggle/input").rglob("vi_asr_project/pyproject.toml")
)
dataset_projects = [path for path in project_candidates if "vi-asr-code" in path.parts]
dataset_zips = [path for path in zip_candidates if "vi-asr-code" in path.parts]
handoff_projects = [
    path for path in project_candidates
    if (path.parent / "artifacts" / "vi_asr_handoff.json").is_file()
]

source_path = None
source_kind = None
if (preferred_project / "pyproject.toml").is_file():
    # Kaggle thường bung ZIP đã upload thành thư mục read-only này.
    source_path, source_kind = preferred_project, "directory"
elif len(dataset_projects) == 1:
    # Kaggle image mới mount theo /kaggle/input/datasets/<owner>/<slug>/...
    source_path, source_kind = dataset_projects[0], "directory"
elif len(dataset_projects) > 1:
    raise RuntimeError(f"Có nhiều thư mục source trong dataset vi-asr-code: {dataset_projects}")
elif preferred_zip.is_file():
    source_path, source_kind = preferred_zip, "zip"
elif len(dataset_zips) == 1:
    source_path, source_kind = dataset_zips[0], "zip"
elif len(dataset_zips) > 1:
    raise RuntimeError(f"Có nhiều source ZIP trong dataset vi-asr-code: {dataset_zips}")
elif len(zip_candidates) == 1:
    source_path, source_kind = zip_candidates[0], "zip"
elif len(zip_candidates) > 1:
    raise RuntimeError(f"Có nhiều source ZIP, chỉ giữ một bản: {zip_candidates}")
elif len(handoff_projects) == 1:
    source_path, source_kind = handoff_projects[0], "directory"
elif not handoff_projects and len(project_candidates) == 1:
    source_path, source_kind = project_candidates[0], "directory"
else:
    raise FileNotFoundError(
        "Không xác định được duy nhất source project. Hãy attach dataset vi-asr-code "
        "hoặc output Notebook 01."
    )

if not (work_dir / "pyproject.toml").is_file():
    if source_kind == "zip":
        with zipfile.ZipFile(source_path, "r") as archive:
            archive.extractall("/kaggle/working")
        print("Đã giải nén source:", source_path)
    else:
        shutil.copytree(source_path, work_dir)
        print("Đã copy source:", source_path)

def sha256_source(path):
    path = Path(path)
    digest = hashlib.sha256()
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    for item in files:
        relative = item.name if path.is_file() else item.relative_to(path).as_posix()
        digest.update(relative.encode("utf-8"))
        digest.update(b"\0")
        with item.open("rb") as stream:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                digest.update(block)
    return digest.hexdigest()

source_sha256 = sha256_source(source_path)
print({"source": str(source_path), "kind": source_kind, "sha256": source_sha256})

os.environ["PYTHONPATH"] = str(work_dir / "src")
if str(work_dir / "src") not in sys.path:
    sys.path.insert(0, str(work_dir / "src"))

%cd /kaggle/working/vi_asr_project
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.in"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import vi_asr
print("vi_asr:", vi_asr.__file__)


In [ ]:
# 3. Clean rebuild — chỉ xóa artifact thuộc pipeline trong /kaggle/working
import shutil
from pathlib import Path

targets = [
    Path("/kaggle/working/artifacts/prepared"),
    Path("/kaggle/working/artifacts/teacher_cache"),
    Path("/kaggle/working/artifacts/audit_report"),
    Path("/kaggle/working/artifacts/tokenizer"),
    Path("/kaggle/working/artifacts/prepared_audio.tar.gz"),
    Path("/kaggle/working/artifacts/vi_asr_handoff.json"),
    Path("/kaggle/working/vi_asr_project/artifacts/tokenizer"),
]
for target in targets:
    resolved = target.resolve()
    assert str(resolved).startswith("/kaggle/working/"), resolved
    if resolved.is_dir():
        shutil.rmtree(resolved)
    elif resolved.exists():
        resolved.unlink()
Path("/kaggle/working/artifacts").mkdir(parents=True, exist_ok=True)
print("Clean rebuild directory is ready.")


In [ ]:
# 4. Stream Hugging Face và materialize đầy đủ train/valid/test
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "vi_asr", "prepare-data",
    "--config", "configs/base.yaml",
    "--paths", "configs/paths/kaggle_prepare.yaml",
], check=True)


In [ ]:
# 5. Xác nhận coverage WAV trước các bước tốn GPU
import json
from pathlib import Path

prepared = Path("/kaggle/working/artifacts/prepared")
split_counts = {}
for split in ("train", "valid", "test"):
    rows = [json.loads(line) for line in (prepared / f"{split}.jsonl").open(encoding="utf-8") if line.strip()]
    missing = [row["audio_relpath"] for row in rows if not (prepared / row["audio_relpath"]).is_file()]
    wav_count = len(list((prepared / "audio" / split).glob("*.wav")))
    split_counts[split] = len(rows)
    print(f"{split:5s}: manifest={len(rows):6d} | wav={wav_count:6d} | missing={len(missing)}")
    assert wav_count == len(rows) and not missing, f"Audio chưa đủ cho split {split}"
assert all(split_counts.values())


In [ ]:
# 6. Huấn luyện SentencePiece BPE chỉ từ train
subprocess.run([
    sys.executable, "-m", "vi_asr", "build-tokenizer",
    "--config", "configs/base.yaml",
    "--paths", "configs/paths/kaggle_prepare.yaml",
], check=True)


In [ ]:
# 7. Tạo PhoBERT content-token cache cho toàn bộ train
subprocess.run([
    sys.executable, "-m", "vi_asr", "cache-teacher",
    "--config", "configs/base.yaml",
    "--paths", "configs/paths/kaggle_prepare.yaml",
    "--split", "train", "--device", "cuda:0",
], check=True)


In [ ]:
# 8. Gate G1: leakage audit và CTC validity
subprocess.run([
    sys.executable, "-m", "vi_asr", "audit",
    "--config", "configs/base.yaml",
    "--paths", "configs/paths/kaggle_prepare.yaml",
    "--report-dir", "/kaggle/working/artifacts/audit_report",
], check=True)


In [ ]:
# 9. Đóng WAV thành ZIP_STORED shards (~512 MiB/shard)
# Mỗi WAV được xóa sau khi ghi thành công, nên không tạo thêm một bản audio 9.4 GiB.
subprocess.run([
    sys.executable, "scripts/kaggle_handoff.py", "build",
    "--prepared-root", "/kaggle/working/artifacts/prepared",
    "--teacher-root", "/kaggle/working/artifacts/teacher_cache",
    "--tokenizer-root", "/kaggle/working/vi_asr_project/artifacts/tokenizer",
    "--audit-root", "/kaggle/working/artifacts/audit_report",
    "--artifact-root", "/kaggle/working/artifacts",
    "--dataset-revision", "3a5b30157034e7eadd5c75fae1a820c6f9383398",
    "--shard-size-mib", "512",
    "--remove-source-audio",
], check=True)


In [ ]:
# 10. Final gate — chỉ Save Version khi cell này PASS
import json
import shutil
from pathlib import Path
from scripts.kaggle_handoff import validate_audio_shards

artifact_root = Path("/kaggle/working/artifacts")
handoff_path = artifact_root / "vi_asr_handoff.json"
handoff = json.loads(handoff_path.read_text(encoding="utf-8"))
assert handoff["schema_version"] == 2

prepared = artifact_root / "prepared"
verified = validate_audio_shards(prepared, verify_hashes=True)
assert verified["files"] == 15023 + 1900 + 2026
assert not list(prepared.rglob("*.wav")), "WAV rời còn sót lại sẽ làm output quá nhiều file"

print(json.dumps({"handoff": handoff, "verified_audio": verified}, indent=2, ensure_ascii=False))
print("Working used:", shutil.disk_usage("/kaggle/working").used / 1024**3, "GiB")

cache_root = Path("/kaggle/working/.cache")
if cache_root.exists():
    shutil.rmtree(cache_root)
    print("Đã xóa download cache khỏi notebook output.")

# Giữ source dưới dạng một ZIP thay vì hàng nghìn file trong project tree.
project_copy = Path("/kaggle/working/vi_asr_project")
if project_copy.exists():
    subprocess.run([
        sys.executable, str(project_copy / "scripts" / "package_project.py"),
        "--output", "/kaggle/working/vi_asr_project.zip",
    ], check=True)
    os.chdir("/kaggle/working")
    shutil.rmtree(project_copy)
    print("Đã thay working project bằng vi_asr_project.zip.")
else:
    assert Path("/kaggle/working/vi_asr_project.zip").is_file()

print("\nPASS: Save Version rồi attach output này làm Input của Notebook 02.")
